# Simulation-driven genetic algorithm

Searches for the extreme (31, 34) graph topologies at r=1.1 -- the size of `avian_r4_l7` --
by evolving them, scoring each candidate by **running the simulation**, not by predicting
it with a regressor.

This notebook **launches** runs and **reads** their results. The loop itself runs as an LSF
job (`moran_process.pipeline.ga_search`), because a real run takes hours and must survive a
dropped kernel or a preemption. Nothing here computes for more than a few seconds.

## How to run one

1. Run the **setup** cell below.
2. In **section 1**, set `THETAS` (which directions to search) and the four size knobs, and
   run the preview cell. It submits nothing; it prints what the launch will cost.
3. Set `PREFIX` to today's date and run the **launch** cell. One LSF driver job per run.
4. Check **section 2** whenever you like. Sections 3-6 read results and work mid-flight.

Everything lands in `simulation_data/ga_runs/<prefix>-theta<NNN>/`, one standard batch
directory per generation.

## A run is one angle

Each run maximizes a weighted sum of the two metrics. They are not comparable as they stand
(`mean_steps` runs 2400-41000, `prob_fixation` 0.09-0.16), so each is first expressed as a
residual from the complete graph and divided by the spread of that residual **among random
(31, 34) graphs**:

$$f \;=\; \underbrace{\cos\theta}_{w_\rho} \, \frac{\rho - \rho_c}{0.00426}
\;+\; \underbrace{\sin\theta}_{w_T} \, \frac{\log(T/T_c)}{0.1586}$$

Both terms are then in "standard deviations away from a typical random graph", the complete
graph scores exactly 0, and since the weights are a unit vector the score is the
**projection** of a graph onto the search direction. The normalizers are constants, not
recomputed per generation: a population-relative scale would drift as the population
converges, so generation 1 and generation 100 would not be on the same axis.

| $\theta$ | seeks | |
|---|---|---|
| 0 | high fixation probability | amplifier |
| 45 | high probability, long time | |
| 90 | long fixation time | |
| 135 | low probability, long time | runs *with* the natural trend |
| 180 | low fixation probability | suppressor |
| 225 | low probability, short time | |
| 270 | short fixation time | |
| 315 | high probability, short time | **the hard one** |

The asymmetry is real: among random (31, 34) graphs the two metrics are *positively*
correlated (+0.35), so graphs that fix more often also take longer. $\theta=315$ asks the
search to break that trend; $\theta=135$ asks it to follow one.

**Direction lives in $\theta$ and nowhere else.** Every run is a maximization, and $\theta$
is normalized to [0, 360), so -45 and 315 are the same run in the same directory.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

import sys

sys.path.insert(0, "src")

from pathlib import Path

import numpy as np
import pandas as pd

from moran_process.pipeline import ga_search
from moran_process.analysis.analysis_utils import ga_io, ga_plots

GA_RUNS_DIR = Path("simulation_data/ga_runs")

# The batch whose 500 random (31, 34) graphs the winners are compared against, and the
# batch the SD normalizers above were measured on.
REFERENCE_BATCH = Path("simulation_data/2026_07_28-respiratory-vs-random-10K-reps-3")

## 1. Launch

**The four size knobs.** A generation holds `pop_size` candidates at generation 0, and
`pop_size * (1 + n_children)` after that -- elites are re-simulated alongside their
children, so a lucky-high score is never carried forward untested.

| knob | what it buys | validation | real run |
|---|---|---|---|
| `generations` | how far the search gets | 15 | 100 |
| `pop_size` | elites kept | 6 | 20 |
| `n_children` | mutants per elite | 4 | 10-20 |
| `n_repeats` | precision of each score | 1e6 | 1e5 |

`n_repeats` is set by **selection efficiency**, the correlation between measured and true
fitness, $\rho = 1/\sqrt{1 + (\mathrm{SEM}/\mathrm{SD})^2}$, since the per-generation
response to selection is proportional to it. Measured: 1e6 gives $\rho \approx 0.99$, 1e5
gives $\approx 0.85$, 1e3 gives $\approx 0.39$. Section 6 checks it after the fact for
free. Note that fewer repeats is not *wrong*, only slower per generation of progress, so
it trades against `generations` and the preview cell prices both.

**Wall clock is mostly latency at small sizes.** ~78 s of every generation is queue wait
and worker startup that does not shrink with the work, on top of ~30 s of simulation. So
15 generations is ~27 min whether the population is 6 or 60.

**Re-running the launch cell RESUMES** an existing run rather than restarting it, which is
what makes an LSF preemption harmless. Change `PREFIX` for a genuinely new launch, or add
`extra_args=["--force"]` to wipe and start over.

Set `NTFY_TOPIC` in your shell before starting the Jupyter server (it is forwarded to the
drivers) and you get one push notification when every run in the launch has ended.

In [ ]:
# --- Cost preview. Submits nothing: edit, re-run, read the numbers. -------------------
# Twelve evenly spaced directions, offset 15 degrees so the four diagonals (45, 135, 225,
# 315) land exactly on sweep points. Those are the interesting corners: 315 is the hard one
# (high probability AND short time, against the +0.35 natural correlation) and 135 the easy
# mirror. A 0-based linspace would straddle them instead.
#
# NOTE endpoint=False. np.linspace(0, 360, 12) includes BOTH ends, so the last direction is
# the same as the first and the spacing is 32.7 rather than 30.
THETAS = 15 + np.arange(12) * 30.0
REPLICATES = 3                        # >1 repeats every direction independently

GENERATIONS = 100
POP_SIZE = 20
N_CHILDREN = 10
N_REPEATS = 100_000

for theta in THETAS:
    w_prob, w_time = ga_search.theta_weights(theta)
    print(
        f"  {ga_search.theta_category(theta)}  w=({w_prob:+.3f}, {w_time:+.3f})  "
        f"{ga_search.quadrant_name(theta)}"
    )
print()

ga_search.preview_launch(
    generations=GENERATIONS,
    pop_size=POP_SIZE,
    n_children=N_CHILDREN,
    n_repeats=N_REPEATS,
    n_runs=len(THETAS) * REPLICATES,
);


In [ ]:
# --- Launch. One driver job per (direction, replicate). ------------------------------
PREFIX = "2026_08_19-sweep12"

# LAUNCHED 2026-08-19 01:40 with exactly the settings above: 12 directions x 3 replicates
# = 36 drivers. The call is commented out so that a "Run All" READS the results instead of
# resubmitting them -- re-running it would resume rather than restart, which is harmless
# but spends 36 driver jobs to discover there is nothing left to do.
#
# Uncomment for a new search (change PREFIX first) or to revive runs that died.

# jobs = ga_search.submit_theta_runs(
#     GA_RUNS_DIR,
#     prefix=PREFIX,
#     thetas=THETAS,
#     replicates=REPLICATES,
#     generations=GENERATIONS,
#     pop_size=POP_SIZE,
#     n_children=N_CHILDREN,
#     n_repeats=N_REPEATS,
#     seed=42,
#     queue="gsla-cpu",             # for the per-generation simulation arrays
#     driver_walltime="8:00",
#     # 1 h, not the 900 s used on the small validation runs. This bounds BOTH the wait for
#     # the array and the wait for its shards to land. At 900 s a single straggler worker
#     # in a saturated queue caused a 101/102-complete generation to be discarded and
#     # re-run, which turned a 3 h launch into a 10 h one.
#     extra_args=["--generation-timeout-s", "3600"],
# )
# jobs


## 2. Progress

Safe to run at any time, including mid-flight: it reads each run's `ga_state.json` and
nothing else. The drivers run detached, so this is how you see where the search is.

In [ ]:
# Runs are found on disk by PREFIX rather than taken from the launch cell's return value:
# the launch call is commented out, and a kernel that still holds `jobs` from an EARLIER
# launch would silently report on those runs instead of these.
#
# load_ga_runs identifies runs by the presence of ga_config.json rather than by name, and
# raises if the prefix matches nothing, so a typo says so instead of looking like a search
# that has not started yet.
ALL_RUNS = ga_io.load_ga_runs(GA_RUNS_DIR, prefix=PREFIX)

ga_io.ga_progress(ALL_RUNS)

# Sections 4 and 5 report on the LAST generation of each run, which for a run still in
# flight is wherever it happens to be right now -- so an unfinished run would appear on a
# "winners" figure with a half-evolved population and quietly understate its direction.
# Set False to include them anyway (useful for watching a search take shape mid-flight).
FINISHED_ONLY = True

RUNS = [r for r in ALL_RUNS
        if not FINISHED_ONLY
        or (ga_io.load_ga_state(r) or {}).get("status") == "finished"]

# Every figure below is ALSO written here as a PNG, so results survive the kernel and can
# be dropped straight into a slide deck. Scoped by PREFIX because the saved filename is
# keyed on the function name and a couple of kwargs, not on the data: without the subfolder
# a second launch would silently overwrite the first launch's figures.
FIGURES_DIR = GA_RUNS_DIR / "figures" / PREFIX
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

skipped = len(ALL_RUNS) - len(RUNS)
print(f"\n{len(RUNS)} of {len(ALL_RUNS)} runs used below"
      + (f"  ({skipped} still running, excluded by FINISHED_ONLY)" if skipped else ""))
print(f"figures -> {FIGURES_DIR}")


In [ ]:
# Anything the driver had to warn about (a generation that came back short and was
# automatically resubmitted). Empty is the expected result.
#
# state["run"] rather than run.name: a run directory's own name is just "rep0" now that
# replicates live under a shared theta directory, which is ambiguous across all 36 runs --
# state["run"] is the flat label ("PREFIX-theta315-rep0") the driver recorded at launch.
for run in RUNS:
    state = ga_io.load_ga_state(run)
    if state and state.get("warnings"):
        print(state.get("run", run), state["warnings"])


## 3. Fitness trajectories

One figure per run: left axis fixation time, right axis fixation probability, solid for
time and dashed for probability, dotted complete-graph baselines as the residual origin.
Both traces are the run's own color, because every run optimizes a *combination* of the two
and neither is incidental.

The shaded band is the spread across surviving elites; the error bars are the standard
error of the measurement. **If the trajectory does not clear its own error bars, the run
found nothing.**

Throughout this notebook, color is the search direction: hue is mapped straight onto
$\theta$, so a run is the same color in every figure and a sweep reads as a color wheel.

In [ ]:
# 36 runs is a lot of trajectory figures. One per DIRECTION (rep0) keeps the notebook
# readable; the replicates are compared as a group in section 4 and 6. Drop the filter to
# draw all 36. Replicates now live as rep0/rep1/rep2 subdirectories under one theta
# directory, so the replicate is the directory's OWN name -- no more parsing a "-repN"
# suffix off a flat run name.
for run in RUNS:
    if run.name == "rep0":
        ga_plots.plot_ga_history(run, figures_dir=FIGURES_DIR)


In [ ]:
# All 36 runs on shared axes, colored by direction. The envelope they trace is the
# reachable range for (31, 34) graphs.
ga_plots.plot_ga_runs_comparison(RUNS, figures_dir=FIGURES_DIR);


## 4. Where the winners sit

The figure the experiment exists to produce. `mutate_graph` preserves both node and edge
count, so every graph the search ever sees is size-matched to `avian_r4_l7`. This asks:
among all connected graphs with the avian lung's node and edge count, what are the
extremes, and where does the real topology sit among them?

**Reading the scatter.** Gray is the 500 random (31, 34) graphs; diamonds are each run's
surviving elites, with its single best outlined in black; stars are the respiratory graphs.
Only the avian graph is (31, 34) -- the mammalian and fish markers carry their own size in
brackets, because they are *not* size-matched and the comparison is looser.

The dashed line through each best winner is that run's **objective iso-line**. Maximizing
a linear objective returns a support point of the achievable set, so a run that worked
leaves the entire random cloud on the losing side of its own line. The table after it puts
a number on the same thing.

In [ ]:
reference_stats = pd.read_csv(REFERENCE_BATCH / "graph_statistics.csv")

# show_hull=True now that a full circle of directions was searched: the outline IS the
# boundary of the achievable region, which is the headline figure of this sweep. Each
# direction's dashed iso-line should be tangent to that boundary at its own winner.
ga_plots.plot_ga_winners_scatter(
    RUNS, reference_stats, show_hull=True, figures_dir=FIGURES_DIR
);


In [ ]:
# The same plane, but the PATH rather than only the destination. One trail per run; each
# point is that generation's mean over its surviving elites. Every run starts from a
# statistically identical random population, so all the trails leave from the same spot in
# the middle of the cloud -- the fan-out is the objective doing its work, not the starting
# conditions differing. Circle = generation 0, diamond = final.
#
# `every=5` subsamples generations. At 36 runs x 100 generations the full trail is 3600
# points and renders as a hairball; every 5th keeps the shape and drops only the jitter.
# show_generation_dots=True marks every generation on the line instead, so its spacing
# shows how much the population moved that generation -- dense once converged, sparse
# while still making progress.
ga_plots.plot_ga_trails(
    RUNS, reference_stats, every=5, figures_dir=FIGURES_DIR, show_generation_dots=True
);

In [ ]:
# Interactive version of the two figures above, in one. Hover any point for the graph that
# produced it; click a legend entry to hide a direction, double-click to isolate it. With
# 12 directions x 3 replicates overlapping, this is the one you actually explore with --
# the static figures are for the slide deck.
#
# No trailing semicolon: a plotly Figure has to be RETURNED to render, and unlike
# matplotlib it is only drawn once.
ga_plots.plot_ga_explorer_plotly(RUNS, reference_stats, trails=True, every=5, show_generation_dots=True)


In [ ]:
# The same check as a number, because "on the losing side" is hard to judge by eye near
# the line: how many of the 500 random graphs beat each winner on that winner's OWN
# objective. Zero means the search has cleared the cloud it is being compared against.
cloud = reference_stats.query(
    "n_nodes == 31 and n_edges == 34 and r == 1.1 and category == 'Random'"
)
winners = ga_io.final_population_stats(RUNS)

rows = []
for run, group in winners.groupby("run"):
    theta = group["theta"].iloc[0]
    best = group.loc[group["rank"].idxmin()]
    cloud_scores = ga_search.weighted_score(
        cloud["prob_fixation"], cloud["mean_steps"], *ga_search.theta_weights(theta)
    )
    rows.append({
        "run": run,
        "theta": theta,
        "seeks": ga_search.quadrant_name(theta),
        "best_score": round(best["weighted"], 3),
        "sem": round(best["weighted_sem"], 3),
        "prob_fixation": round(best["prob_fixation"], 5),
        "mean_steps": round(best["mean_steps"], 1),
        "cloud_beats_it": int((cloud_scores > best["weighted"]).sum()),
    })
pd.DataFrame(rows).sort_values("theta").reset_index(drop=True)

## 5. The winning topologies

What the extremes actually look like. Measured, not predicted.

In [ ]:
# Every run's winner on one sheet, replicates side by side in each row. This is where the
# convergent-evolution question gets answered: same direction, independent seeds -- did
# they arrive at the same KIND of graph?
ga_plots.plot_elite_grid(RUNS, per_row=3, figures_dir=FIGURES_DIR);


In [ ]:
# Zoom in on ONE direction: its whole final population, not just the winner.
#
#   THETA        which direction, in degrees. Normalized, so -45 and 315 are the same run.
#   replicate    0/1/2, or None for every replicate of that direction stacked together.
#   n            how many elites per run (None = all 20).
#
# A direction whose 20 elites are all the same shape has converged; a mixed bag has not.
THETA = 315

ga_plots.plot_population(
    GA_RUNS_DIR, PREFIX, theta=THETA, replicate=0, n=10, figures_dir=FIGURES_DIR
);

# The graph objects themselves, if you want to measure something rather than look at it:
#   population = ga_io.load_elite_population(GA_RUNS_DIR / f"{PREFIX}-theta315" / "rep0")
#   population[0].graph          # the networkx object
#   ga_io.final_elite_properties(RUNS)   # every elite joined to its structural properties


## 6. Was selection still working?

`n_repeats` exists to make the ranking mean something. This checks whether it did, after
the fact and at no simulation cost: every input is already in `ga_history.csv`.

**Left panel: is rho near 1?** Selection ranks candidates on a *measured* score, so what it
responds to is the true value plus noise, and the response is proportional to
$\rho = 1/\sqrt{1 + (\mathrm{SEM}/\mathrm{SD})^2}$. Below the shaded line at 0.5, more
repeats buy more than more generations do.

**Right panel: why did it move?** Solid is the between-graph spread, dashed the measurement
error, each relative to its own first generation. Rho can fall two ways. The population
converges and the real spread between graphs collapses (solid drops) -- that is the search
succeeding, not a problem. Or the noise floor rises (dashed climbs), which happens to any
run pushing fixation time up, because the SEM of a mean is proportional to that mean:
**such a run inflates its own noise floor in step with its own signal.**

In [ ]:
ga_plots.plot_selection_efficiency(RUNS, figures_dir=FIGURES_DIR);


### 6b. Was selection still *buying* anything?

Section 6 asks whether the ranking was trustworthy. This asks the complementary question:
whether that ranking was still producing improvement. A search can have excellent $\rho$
and be completely stuck, if every child is already as good as its parent.

**Left: gain per generation.** The change in the surviving elites' mean score, smoothed.
The gray band is the measurement noise on that difference. A curve inside the band is a
population whose apparent movement cannot be told apart from re-measuring the same graphs.

**Right: elites displaced by their own children.** The mechanical version of the same
question, and it needs no error bar: if nothing displaces an incumbent, nothing is
happening. Read the two together, because the interesting case is when they *disagree* --
high turnover with zero gain means elites are losing their places to measurement noise
rather than to genuinely better children, which is churn, not progress.


In [ ]:
# show_runs=False averages the replicates into one line per direction, which is easier to
# read on a slide; True (the default) draws all 36 so replicate disagreement stays visible.
ga_plots.plot_selection_response(RUNS, smooth=5, figures_dir=FIGURES_DIR);
